# Freight Cost Prediction

Predict freight cost from vendor-invoice and purchase-order attributes. The workflow uses a chronological 80/20 holdout to simulate forecasting future transactions.

**Model:** ExtraTreesRegressor  
**Metrics:** MAE, RMSE, R²

## 1. Setup
The database is intentionally kept local. Run this notebook from the repository after placing `data.db` at the project root.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from src.data import load_table
from src.freight_model import build_model, FEATURES, TARGET

In [ ]:
df = load_table('vendor_invoice').copy()
df['PODate'] = pd.to_datetime(df['PODate'], errors='coerce')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['days_po_to_invoice'] = (df['InvoiceDate'] - df['PODate']).dt.days
df['po_month'] = df['PODate'].dt.month
df['po_day_of_week'] = df['PODate'].dt.dayofweek
df = df.dropna(subset=[TARGET, 'PODate']).sort_values('PODate').reset_index(drop=True)
df.shape

## 2. Quick EDA
Check the target distribution and relationships before modeling.

In [ ]:
df[['Dollars', 'Quantity', 'Freight', 'days_po_to_invoice']].describe().round(2)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Dollars'], df['Freight'], alpha=0.4)
plt.xlabel('Invoice Dollars')
plt.ylabel('Freight Cost')
plt.title('Invoice Value vs Freight Cost')
plt.show()

## 3. Chronological train/test split
A random split can mix future observations into training data. Sorting by PO date and holding out the latest 20% gives a more realistic estimate for future prediction.

In [ ]:
cut = int(len(df) * 0.80)
train = df.iloc[:cut].copy()
test = df.iloc[cut:].copy()
print(f'Train rows: {len(train):,}')
print(f'Test rows:  {len(test):,}')
print(f'Train period: {train.PODate.min().date()} to {train.PODate.max().date()}')
print(f'Test period:  {test.PODate.min().date()} to {test.PODate.max().date()}')

## 4. Train ExtraTrees model

In [ ]:
model = build_model()
model.fit(train[FEATURES], train[TARGET])
pred = model.predict(test[FEATURES])
metrics = {
    'MAE': mean_absolute_error(test[TARGET], pred),
    'RMSE': mean_squared_error(test[TARGET], pred) ** 0.5,
    'R2': r2_score(test[TARGET], pred),
}
pd.Series(metrics).round(3)

## 5. Actual vs predicted
The evaluation below is performed only on the chronological holdout.

In [ ]:
evaluation = pd.DataFrame({'actual_freight': test[TARGET].values, 'predicted_freight': pred})
evaluation['absolute_error'] = (evaluation['actual_freight'] - evaluation['predicted_freight']).abs()
evaluation.head(10)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(evaluation['actual_freight'], evaluation['predicted_freight'], alpha=0.4)
lims = [evaluation.min().min(), evaluation.max().max()]
plt.plot(lims, lims, linestyle='--')
plt.xlabel('Actual Freight')
plt.ylabel('Predicted Freight')
plt.title('Actual vs Predicted Freight')
plt.show()

## 6. Conclusion
The model is evaluated using a time-aware holdout rather than a random split. The same preprocessing and estimator are implemented in `src/freight_model.py` so the workflow can also be executed through `scripts/train_freight.py`.